In [0]:
%pip install geopandas osmnx shapely plotly

In [0]:
import pandas as pd
import json
import plotly.express as px
import geopandas as gpd
from shapely.geometry import Polygon
import osmnx as ox
import base64

# 1. Load data from Databricks Delta table
spark_df = spark.table("dbr_dev.artemzharkov10_gold.gold_realtime_road_hazard")
df = spark_df.toPandas()

dt_series = pd.to_datetime(df['weather_time'])
df['time_str'] = dt_series.dt.floor('15min').dt.strftime('%Y-%m-%d %H:%M')
df = df.sort_values('time_str')

# 2. Create grid geometry (executed once)
d_lon = 0.52
d_lat = 0.315
polygons = []
ids = []

unique_grids = df[['ID', 'longitude', 'latitude']].drop_duplicates()

for _, row in unique_grids.iterrows():
    lon, lat = row['longitude'], row['latitude']
    half_lon, half_lat = d_lon / 2, d_lat / 2
    
    poly = Polygon([
        (lon - half_lon, lat + half_lat),
        (lon + half_lon, lat + half_lat),
        (lon + half_lon, lat - half_lat),
        (lon - half_lon, lat - half_lat)
    ])
    polygons.append(poly)
    ids.append(row['ID'])

grid_gdf = gpd.GeoDataFrame({'ID': ids, 'geometry': polygons}, crs="EPSG:4326")

# 3. Clip grid to Poland's boundaries and OPTIMIZE GEOMETRY
poland_gdf = ox.geocode_to_gdf("Poland")
clipped_grid = gpd.clip(grid_gdf, poland_gdf)

# Simplify boundaries to significantly reduce JSON file size
clipped_grid['geometry'] = clipped_grid['geometry'].simplify(tolerance=0.02)
geojson_grid = json.loads(clipped_grid.to_json())

# 4. Generate Soil Temperature Map
fig_temp = px.choropleth_mapbox(
    df,
    geojson=geojson_grid,
    locations='ID',                  
    featureidkey='properties.ID',    
    color='soil_temperature_c',        
    color_continuous_scale='RdYlBu_r',
    range_color=[-15, 30],
    mapbox_style="carto-positron",   
    zoom=5.5,
    center={"lat": 52.0693, "lon": 19.4803},
    opacity=0.4,
    animation_frame='time_str', 
    title='Soil Temperature Map',
    labels={'soil_temperature_c': 'Soil Temp. (°C)'}
)

fig_temp.update_traces(marker_line_width=0)
fig_temp.update_layout(margin={"r":0,"t":40,"l":0,"b":0})

# 5. Generate Precipitation Map
fig_precip = px.choropleth_mapbox(
    df,
    geojson=geojson_grid,
    locations='ID',                  
    featureidkey='properties.ID',    
    color='precipitation_mm',        
    color_continuous_scale='Blues',  
    range_color=[0, 2],
    mapbox_style="carto-positron",   
    zoom=5.5,
    center={"lat": 52.0693, "lon": 19.4803},
    opacity=0.4,
    animation_frame='time_str',    
    title='Precipitation Map',
    labels={'precipitation_mm': 'Precipitation (mm)'}
)

fig_precip.update_traces(marker_line_width=0)
fig_precip.update_layout(margin={"r":0,"t":40,"l":0,"b":0})

# 6. Convert maps to Base64 and generate download links (Bypassing blocks)
html_temp = fig_temp.to_html(include_plotlyjs='cdn')
html_precip = fig_precip.to_html(include_plotlyjs='cdn')

b64_temp = base64.b64encode(html_temp.encode('utf-8')).decode('utf-8')
b64_precip = base64.b64encode(html_precip.encode('utf-8')).decode('utf-8')

displayHTML(f"""
    <div style="font-family: Arial; padding: 15px; border: 1px solid #ccc; border-radius: 5px;">
        <h3>Interactive Maps Generated</h3>
        <p>Databricks security policy blocks direct viewing. Click the links below to download the files, then open them in your browser.</p>
        <ul style="list-style-type: none; padding-left: 0;">
            <li style="margin-bottom: 10px;">
                <a href="data:text/html;base64,{b64_temp}" download="temperature_map.html" style="text-decoration: none; font-size: 16px; font-weight: bold; color: #0055ff;">
                    📥 Download Temperature Map (temperature_map.html)
                </a>
            </li>
            <li>
                <a href="data:text/html;base64,{b64_precip}" download="precipitation_map.html" style="text-decoration: none; font-size: 16px; font-weight: bold; color: #0055ff;">
                    📥 Download Precipitation Map (precipitation_map.html)
                </a>
            </li>
        </ul>
    </div>
""")

In [0]:
import pandas as pd
import json
import plotly.express as px
import geopandas as gpd
from shapely.geometry import Polygon
import osmnx as ox
import base64

# 1. Load data from Databricks Delta table
spark_df = spark.table("dbr_dev.artemzharkov10_gold.gold_realtime_road_hazard")
df = spark_df.toPandas()

dt_series = pd.to_datetime(df['weather_time'])
df['time_str'] = dt_series.dt.floor('15min').dt.strftime('%Y-%m-%d %H:%M')
df = df.sort_values('time_str')

# 2. Create grid geometry (executed once)
d_lon = 0.52
d_lat = 0.315
polygons = []
ids = []

unique_grids = df[['ID', 'longitude', 'latitude']].drop_duplicates()

for _, row in unique_grids.iterrows():
    lon, lat = row['longitude'], row['latitude']
    half_lon, half_lat = d_lon / 2, d_lat / 2
    
    poly = Polygon([
        (lon - half_lon, lat + half_lat),
        (lon + half_lon, lat + half_lat),
        (lon + half_lon, lat - half_lat),
        (lon - half_lon, lat - half_lat)
    ])
    polygons.append(poly)
    ids.append(row['ID'])

grid_gdf = gpd.GeoDataFrame({'ID': ids, 'geometry': polygons}, crs="EPSG:4326")

# 3. Clip grid to Poland's boundaries and OPTIMIZE GEOMETRY
poland_gdf = ox.geocode_to_gdf("Poland")
clipped_grid = gpd.clip(grid_gdf, poland_gdf)
clipped_grid['geometry'] = clipped_grid['geometry'].simplify(tolerance=0.02)
geojson_grid = json.loads(clipped_grid.to_json())

# 4. Generate Hazard Risk Map
fig_risk = px.choropleth_mapbox(
    df,
    geojson=geojson_grid,
    locations='ID',                  
    featureidkey='properties.ID',    
    color='hazard_risk',        
    color_continuous_scale='Turbo',  
    range_color=[df['hazard_risk'].min(), df['hazard_risk'].max()],
    mapbox_style="carto-positron",   
    zoom=5.5,
    center={"lat": 52.0693, "lon": 19.4803},
    opacity=0.5,
    animation_frame='time_str', 
    title='Road Hazard Risk Map',
    labels={'hazard_risk': 'Hazard Risk'},
    hover_data={'weather_claster': True, 'soil_temperature_c': True, 'precipitation_mm': True} 
)

fig_risk.update_traces(marker_line_width=0)
fig_risk.update_layout(margin={"r":0,"t":40,"l":0,"b":0})

# 5. Convert map to Base64 and generate download link
html_risk = fig_risk.to_html(include_plotlyjs='cdn')
b64_risk = base64.b64encode(html_risk.encode('utf-8')).decode('utf-8')

displayHTML(f"""
    <div style="font-family: Arial; padding: 15px; border: 1px solid #ccc; border-radius: 5px;">
        <h3>Interactive Risk Map Generated</h3>
        <ul style="list-style-type: none; padding-left: 0;">
            <li>
                <a href="data:text/html;base64,{b64_risk}" download="hazard_risk_map.html" style="text-decoration: none; font-size: 16px; font-weight: bold; color: #d9381e;">
                    📥 Download Hazard Risk Map (hazard_risk_map.html)
                </a>
            </li>
        </ul>
    </div>
""")